In [ ]:
# =============================================================================
# 模块 1: 全局配置、初始化与模型加载
# =============================================================================

# --- 1. 核心库导入 ---
import os
import re
import json
import glob
import shutil
import py3Dmol
from itertools import combinations

# --- 数据处理与科学计算库 ---
import pandas as pd
import numpy as np
import torch
import scipy.spatial.distance

# --- HuggingFace Transformers ---
from transformers import AutoTokenizer, AutoModel

# --- RDKit 与 PyG ---
from rdkit import Chem, RDLogger
from rdkit.Geometry import Point3D 
from torch_geometric.data import Data

# --- Jupyter 与 TQDM ---
from IPython.display import display
from tqdm.notebook import tqdm

# --- [自定义工具箱] ---
import utils 

# --- 2. 固定全局随机种子 ---
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f"全局随机种子已固定为: {RANDOM_SEED}")

# =============================================================================
#                            全局超参数与文件路径配置
# =============================================================================

# --- Embedding 维度设置 ---
RAW_EMBEDDING_DIM = 768 
TARGET_EMBEDDING_DIM = 64  
EMBEDDING_DIM = TARGET_EMBEDDING_DIM 

# --- 处理范围控制 ---
NUM_COMPLEXES_TO_PROCESS = None 

# ---  输入路径定义 ---
INPUT_ROOT = os.path.join('datas', 'work_datas', 'SE3TD')   

# --- Task A/B 输入路径 (用户放置原始 PDB 的目录) ---
TASK_A_INPUT_DIR = os.path.join(INPUT_ROOT, 'TASK_A')
TASK_B_INPUT_DIR = os.path.join(INPUT_ROOT, 'TASK_B')

# ---  输出路径定义 ---
OUTPUT_ROOT = os.path.join('processed_data')

# --- Task A/B 输出路径 ---
TASK_A_OUTPUT_DIR = os.path.join(OUTPUT_ROOT, 'TASK_A')
TASK_B_OUTPUT_DIR = os.path.join(OUTPUT_ROOT, 'TASK_B')

for d in [TASK_A_INPUT_DIR, TASK_B_INPUT_DIR,
          TASK_A_OUTPUT_DIR, TASK_B_OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

# --- 任务模式选择 ---
# "TASK_A": 有参考配体，从 TASK_A 文件夹读取复合物 PDB
# "TASK_B": 无参考配体，从 TASK_B 文件夹读取纯蛋白 PDB
TASK_MODE = "TASK_A"   

print(f"输入根目录: {INPUT_ROOT}")
print("输出目录结构已准备就绪。")

# ---  共享资源文件路径 ---
VOCAB_FILE = os.path.join(OUTPUT_ROOT, 'fragment_vocabulary_64_dynamic_margin.json')
NORM_STATS_FILE = os.path.join(OUTPUT_ROOT, 'normalization_stats_2.9w.json')


# =============================================================================
#                      ChemBERTa 模型初始化与降维矩阵
# =============================================================================

HF_MODEL_NAME = "seyonec/ChemBERTa-Zinc-Base-v1"
LOCAL_MODEL_PATH = os.path.join(os.getcwd(), "ChemBERTa_Local")

if not os.path.exists(LOCAL_MODEL_PATH):
    print(f"[ChemBERTa] 警告: 本地模型路径不存在: {LOCAL_MODEL_PATH}")
else:
    print(f"[ChemBERTa] 正在加载本地模型: {LOCAL_MODEL_PATH}...")

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
model = AutoModel.from_pretrained(LOCAL_MODEL_PATH)
model.eval() 
print(f"[ChemBERTa] 模型加载完成。")

if TARGET_EMBEDDING_DIM < RAW_EMBEDDING_DIM:
    print(f"[Embedding] 初始化随机投影矩阵: {RAW_EMBEDDING_DIM} -> {TARGET_EMBEDDING_DIM}")
    torch.manual_seed(RANDOM_SEED) 
    np.random.seed(RANDOM_SEED)
    _proj = torch.randn(RAW_EMBEDDING_DIM, TARGET_EMBEDDING_DIM)
    PROJECTION_MATRIX = _proj / torch.norm(_proj, dim=0)
else:
    print(f"[Embedding] 目标维度 >= 原生维度，不进行压缩。")
    PROJECTION_MATRIX = None


# =============================================================================
#                              状态加载
# =============================================================================

try:
    with open(VOCAB_FILE, 'r', encoding='utf-8') as f:
        vocab_data = json.load(f)
    vocabulary = set(vocab_data.keys())
    print(f"成功从 '{VOCAB_FILE}' 加载已有词汇表，当前大小: {len(vocabulary)}")
except (FileNotFoundError, json.JSONDecodeError):
    vocab_data = {}
    vocabulary = set()
    print(f"提示: 词汇表文件 '{VOCAB_FILE}' 未找到或为空，将在运行中创建。")

try:
    with open(NORM_STATS_FILE, 'r', encoding='utf-8') as f:
        norm_stats = json.load(f)
    print(f"成功从 '{NORM_STATS_FILE}' 加载已有归一化统计量。")
except (FileNotFoundError, json.JSONDecodeError):
    norm_stats = {}
    print(f"提示: 归一化文件 '{NORM_STATS_FILE}' 未找到，可能影响特征归一化结果。")

FEATURE_COLUMNS = [
    'avg_ecc', 'ecc_range',
    'avg_logp', 'avg_tpsa', 'avg_charge', 
    'HBA', 'HBD', 'Aromatic', 'Hydrophobe', 'PosIonizable', 'NegIonizable',
    'max_dist_3d', 'max_angle_3d', 'max_plane_angle_3d'
]

ANGLE_FEATURES = {'max_angle_3d': 180.0, 'max_plane_angle_3d': 180.0}
COUNT_FEATURES = {'HBA', 'HBD', 'Aromatic', 'Hydrophobe', 'PosIonizable', 'NegIonizable'}

print("-" * 50)
print("模块 1: 初始化完成。")
# =============================================================================
# 模块 2: 统一辅助工具箱 
# =============================================================================
def clean_protein_from_pdb(pdb_path):
      """
      从 PDB 文件中只保留 ATOM 记录（蛋白质），去除所有 HETATM（配体、离子、水等）。
      返回: (protein_mol, protein_lines)
        - protein_mol: RDKit Mol (纯蛋白)
        - protein_lines: list[str] (原始 ATOM 行，用于后续保存 PDB)
      """
      protein_lines = []
      with open(pdb_path, 'r', encoding='utf-8') as f:
          for line in f:
              if line.startswith('ATOM'):
                  protein_lines.append(line)

      if not protein_lines:
          return None, []

      # 用 protein_lines 构建 RDKit Mol
      pdb_block = ''.join(protein_lines) + 'END\n'
      mol = Chem.MolFromPDBBlock(pdb_block, removeHs=True, sanitize=False)
      if mol is not None:
          mol = utils.advanced_rescue_valence_errors(mol)
          if mol is not None:
              try:
                  mol.UpdatePropertyCache(strict=False)
                  Chem.SanitizeMol(mol, sanitizeOps=Chem.SANITIZE_SYMMRINGS |
                                                    Chem.SANITIZE_SETCONJUGATION |
                                                    Chem.SANITIZE_SETHYBRIDIZATION)
              except:
                  pass

      return mol, protein_lines

def get_chemberta_embedding(smiles):
    if not smiles or smiles == "":
        return torch.zeros(TARGET_EMBEDDING_DIM).tolist()
    try:
        inputs = tokenizer(smiles, return_tensors="pt", padding=True, truncation=True, max_length=512)
        with torch.no_grad():
            outputs = model(**inputs)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        if PROJECTION_MATRIX is not None:
            final_embedding = torch.matmul(cls_embedding, PROJECTION_MATRIX)
        else:
            final_embedding = cls_embedding
        return final_embedding.squeeze().tolist()
    except Exception as e:
        return torch.zeros(TARGET_EMBEDDING_DIM).tolist()


def save_vocabulary_state(current_vocab_set, vocab_dict, vocab_file):
    existing_smiles = set(vocab_dict.keys())
    new_smiles_to_add = current_vocab_set - existing_smiles
    if new_smiles_to_add:
        sorted_new_smiles = sorted(list(new_smiles_to_add))
        for smi in sorted_new_smiles:
            if smi not in vocab_dict:
                vocab_dict[smi] = get_chemberta_embedding(smi)
        tqdm.write(f"  [保存] 词汇表新增了 {len(sorted_new_smiles)} 个片段。")

    if "<UNK>" not in vocab_dict:
        vocab_dict["<UNK>"] = torch.zeros(TARGET_EMBEDDING_DIM).tolist()
    
    temp_dict = vocab_dict.copy()
    unk_embedding = temp_dict.pop("<UNK>")
    sorted_items = sorted(temp_dict.items())

    try:
        with open(vocab_file, 'w', encoding='utf-8') as f:
            f.write("{\n")
            f.write(f'  "<UNK>": {json.dumps(unk_embedding)}')
            if sorted_items:
                f.write(",\n")
                for i, (smi, vec) in enumerate(sorted_items):
                    line = f'  {json.dumps(smi, ensure_ascii=False)}: {json.dumps(vec)}'
                    if i < len(sorted_items) - 1:
                        f.write(f"{line},\n")
                    else:
                        f.write(f"{line}\n")
            else:
                f.write("\n")
            f.write("}")
        print(f"--- 词汇表报告: 已更新并保存至 {vocab_file} (总计: {len(sorted_items) + 1}) ---")
    except Exception as e:
        print(f"保存词汇表时发生错误: {e}")


def process_single_fragment(fragment, origin_type, pdb_id, id_start_offset):
    """
    [修订版 - 适配 utils 更新] 支持返回丢弃计数，支持离子保留
    返回: (shards_list, error_msg, dropped_count)
    """
    try:
        # 1. 初步分离（用于判断是否纯离子）
        main_mol_candidate, ion_fragments_candidate = utils.rule1_separate_ions(fragment)
        results_storage = []
        dropped_count = 0 

        # --- 分支 A: 纯离子片段 (无有机主体) ---
        if main_mol_candidate.GetNumAtoms() == 0:
            d_count = utils.calculate_descriptors_and_store(
                storage=results_storage, mol_data={'IDs': pdb_id, 'Smiles': '[Ion Fragment]'},
                original_mol_with_conf=fragment, main_mol_for_matching=main_mol_candidate,
                final_covalent_fragments=[], ion_fragments=ion_fragments_candidate,
                original_eccentricities=[], original_logp_contribs=[],
                original_tpsa_contribs=[], original_charges=[]
            )
            dropped_count += d_count
            
        # --- 分支 B: 有机分子 (需预处理和切割) ---
        else:
            # [关键修改] pre_calc_data 现在即使主分子失败也会返回字典
            pre_calc_data = utils.prepare_molecule_for_cutting(fragment)
            
            if not pre_calc_data: 
                return [], "预处理致命错误", 1
            
            main_mol = pre_calc_data['main_mol_for_matching']
            ion_fragments = pre_calc_data['ion_fragments'] # 获取预处理阶段再次确认的离子
            
            final_covalent_shards = []
            
            # [关键修改] 检查 main_mol 是否存在
            if main_mol is not None and main_mol.GetNumAtoms() > 0:
                final_covalent_shards = utils.execute_fragmentation_pipeline(main_mol, pre_calc_data['bond_info'])
            else:
                # 主分子重构失败
                if main_mol is None and not ion_fragments:
                     return [], "主分子重构失败且无离子", 1
                
                # 如果主分子丢了，计数+1
                if main_mol is None:
                     dropped_count += 1

            # [关键修改] 传入可能为 None 的 main_mol (utils 内部已做容错处理)
            d_count = utils.calculate_descriptors_and_store(
                storage=results_storage, mol_data={'IDs': pdb_id, 'Smiles': Chem.MolToSmiles(fragment)},
                original_mol_with_conf=fragment, 
                main_mol_for_matching=main_mol if main_mol else fragment, # Fallback 防止 None 传参报错
                final_covalent_fragments=final_covalent_shards, 
                ion_fragments=ion_fragments,
                original_eccentricities=pre_calc_data['original_eccentricities'],
                original_logp_contribs=pre_calc_data['original_logp_contribs'],
                original_tpsa_contribs=pre_calc_data['original_tpsa_contribs'],
                original_charges=pre_calc_data['original_charges']
            )
            dropped_count += d_count

        # 3. 结果组装
        if results_storage:
            shards_list = []
            processed_result = results_storage[0]
            temp_results = processed_result['results'] + processed_result['ions']
            for i, shard_info in enumerate(temp_results):
                shard_info['origin'] = origin_type
                shard_info['Shard ID'] = id_start_offset + i
                shards_list.append(shard_info)
            return shards_list, None, dropped_count 
            
        return [], "未生成结果", dropped_count
        
    except Exception as e:
        return [], str(e), 1
    
def load_molecules_from_file(file_path):
    """
    1. removeHs=True (解决化合价错误的关键)
    2. Partial Sanitization (跳过 Valence 检查)
    3. 引入底层高级化合价抢救机制
    """
    if not os.path.exists(file_path):
        return [], None
    
    file_ext = os.path.splitext(file_path)[1].lower()
    mols = []
    
    lg = RDLogger.logger()
    lg.setLevel(RDLogger.CRITICAL) 
    
    try:
        raw_mols = []
        if file_ext == '.pdb':
            mol = Chem.MolFromPDBFile(file_path, removeHs=True, sanitize=False)
            if mol: raw_mols.append(mol)
        elif file_ext == '.sdf':
            supplier = Chem.SDMolSupplier(file_path, removeHs=True, sanitize=False)
            if supplier:
                raw_mols = [m for m in supplier if m]
        elif file_ext == '.mol':
            mol = Chem.MolFromMolFile(file_path, removeHs=True, sanitize=False)
            if mol: raw_mols.append(mol)
            
        for m in raw_mols:
            # ===============================================================
            # [核心修复：在刚读入内存的第一时间，立刻呼叫“救护车”！]
            # 无论是 Pocket 的 PDB 还是配体的 SDF，只要有畸形，统统切断治愈
            # ===============================================================
            if m is not None:
                m = utils.advanced_rescue_valence_errors(m)
                
            if m is None:
                continue
                
            try:
                # 尝试更新属性，允许错误
                m.UpdatePropertyCache(strict=False)
                # 跳过 Valence 检查，仅计算环、共轭、杂化
                Chem.SanitizeMol(m, sanitizeOps=Chem.SANITIZE_SYMMRINGS | 
                                                Chem.SANITIZE_SETCONJUGATION | 
                                                Chem.SANITIZE_SETHYBRIDIZATION)
                mols.append(m)
            except:
                pass 
                
    except Exception:
        pass
    finally:
        lg.setLevel(RDLogger.INFO)
        
    return mols, file_ext
print("模块 2: 辅助函数定义完成 (已启用 removeHs=True 与 宽松检查)。")


In [ ]:
# =============================================================================
# 模块 3: 文件扫描与任务分发逻辑
# =============================================================================

print("\n" + "="*50)
print("开始执行文件扫描与任务分发...")

# --- 根据 TASK_MODE 扫描对应目录 ---
if TASK_MODE == "TASK_A":
    scan_dir = TASK_A_INPUT_DIR
    print(f"当前模式: TASK_A (有参考配体)")
elif TASK_MODE == "TASK_B":
    scan_dir = TASK_B_INPUT_DIR
    print(f"当前模式: TASK_B (无参考配体)")
else:
    raise ValueError(f"无效的 TASK_MODE: '{TASK_MODE}'，请设为 'TASK_A' 或 'TASK_B'")

task_pdb_files = sorted(glob.glob(os.path.join(scan_dir, '*.pdb')))
task_ids = [os.path.splitext(os.path.basename(f))[0] for f in task_pdb_files]

if NUM_COMPLEXES_TO_PROCESS and isinstance(NUM_COMPLEXES_TO_PROCESS, int):
    print(f"注意: 全局限制生效，仅处理前 {NUM_COMPLEXES_TO_PROCESS} 个条目。")
    task_ids = task_ids[:NUM_COMPLEXES_TO_PROCESS]
    task_pdb_files = task_pdb_files[:NUM_COMPLEXES_TO_PROCESS]

print("-" * 30)
print(f"文件扫描完成，任务队列已生成:")
print(f"  [{TASK_MODE}] 扫描目录: {scan_dir}")
print(f"  [{TASK_MODE}] 发现 PDB 文件: {len(task_ids)} 个")
print("-" * 30)
print("模块 3: 任务分发完成。")
print("="*50)

In [ ]:
# =============================================================================
# 模块 4 (重构): 交互式复合物处理流程 (Survival / Creative 双模式)
# =============================================================================
# 依赖: ipywidgets + py3Dmol
#   pip install ipywidgets py3Dmol

import ipywidgets as widgets
from IPython.display import display, clear_output
import time
import html
from jupyter_ui_poll import ui_events

print("\n" + "="*50)
print("开始执行交互式复合物处理...")

if not task_ids:
    print("没有发现需要处理的 PDB。跳过模块 4。")
else:
    # 准备归一化参数
    final_norm_params = {}
    for prop, stats in norm_stats.items():
        count, mean, M2 = stats['count'], stats['mean'], stats['M2']
        std = np.sqrt(M2 / count) if count >= 2 else 0.0
        final_norm_params[prop] = {'mean': mean, 'std': std}

    unk_embedding = vocab_data.get("<UNK>", torch.zeros(TARGET_EMBEDDING_DIM).tolist())

    # =================================================================
    # 辅助函数: 清理干扰项 (水分子、离子、酸根等，保留金属元素)
    # =================================================================
    INTERFERENCE_RESIDUES = {
        # 水分子
        'HOH', 'WAT', 'H2O', 'DOD', 'D2O',
        # 常见酸根/离子残基
        'SO4', 'PO4', 'NO3', 'CO3', 'CL', 'BR', 'IOD', 'FLC',
        # 常见缓冲液/溶剂/添加剂
        'GOL', 'EDO', 'PEG', 'PGE', 'MPD', 'DMS', 'ACE', 'NH2',
        'BME', 'MES', 'TRS', 'EPE', 'FMT', 'ACT', 'IMD', 'IPA',
        # 注意：金属元素不在此列表中，它们被保留
    }

    def clean_interference_from_candidates(candidates):
        """
        从配体候选列表中移除水分子、离子、酸根等干扰项，但保留金属元素。
        返回: (cleaned_candidates, removed_items)
        """
        cleaned = []
        removed = []
        for cand in candidates:
            res_name = cand['resName'].strip().upper()
            if res_name in INTERFERENCE_RESIDUES:
                removed.append(cand)
            else:
                cleaned.append(cand)
        return cleaned, removed

    # =================================================================
    # 辅助函数: 归一化特征计算 (共用)
    # =================================================================
    def compute_normalized_features(shard):
        """计算单个 shard 的归一化节点特征向量"""
        other_features = []
        for prop in FEATURE_COLUMNS:
            value = shard.get(prop, 0.0)
            if prop in final_norm_params:
                m, s = final_norm_params[prop]['mean'], final_norm_params[prop]['std']
                if prop in ANGLE_FEATURES:
                    norm_val = float(np.clip(value / ANGLE_FEATURES[prop], 0.0, 1.0))
                elif prop in COUNT_FEATURES:
                    safe_std = max(s, 1.0)
                    norm_val = float(np.clip((value - m) / safe_std, -6.0, 6.0))
                else:
                    safe_std = s if s > 1e-5 else 1.0
                    norm_val = float(np.clip((value - m) / safe_std, -6.0, 6.0))
                other_features.append(norm_val)
            else:
                other_features.append(value)
        return other_features

    def get_shard_embedding(shard):
        """获取 shard 对应的 ChemBERTa embedding"""
        clean_smi = "<UNK>"
        if 'clean_mol' in shard and shard['clean_mol']:
            try: clean_smi = Chem.MolToSmiles(shard['clean_mol'], canonical=True)
            except: pass
        elif 'smiles' in shard:
            clean_smi = shard['smiles']

        if clean_smi not in vocab_data:
            if clean_smi == "<UNK>":
                vocab_data["<UNK>"] = torch.zeros(TARGET_EMBEDDING_DIM).tolist()
            else:
                vocab_data[clean_smi] = get_chemberta_embedding(clean_smi)
        return vocab_data.get(clean_smi, unk_embedding)

    # =================================================================
    # 内部辅助函数: 组装合并的 PyG Data (Survival 模式: 口袋+配体)
    # =================================================================
    def assemble_combined_pyg_data(pocket_shards, ligand_shards, all_edge_attrs,
                                      meb_center, complex_id):
        # --- 过滤 centroid_3d/ref_coord_3d 为 None 的无效 shard ---
        valid_pocket = [s for s in pocket_shards if s.get('centroid_3d') is not None and s.get('ref_coord_3d') is not None]
        valid_ligand = [s for s in ligand_shards if s.get('centroid_3d') is not None and s.get('ref_coord_3d') is not None]
        if len(valid_pocket) < len(pocket_shards) or len(valid_ligand) < len(ligand_shards):
            print(f"  [警告] 过滤无3D坐标碎片: 口袋 -{len(pocket_shards)-len(valid_pocket)}, 配体 -{len(ligand_shards)-len(valid_ligand)}")
        old_to_new = {}
        for new_idx, shard in enumerate(valid_pocket + valid_ligand):
            old_to_new[shard['Shard ID']] = new_idx
        all_edge_attrs = {(old_to_new[u], old_to_new[v]): t for (u,v), t in all_edge_attrs.items() if u in old_to_new and v in old_to_new}
        pocket_shards, ligand_shards = valid_pocket, valid_ligand
        # --- 过滤结束 ---

        all_shards = pocket_shards + ligand_shards
        n_pocket = len(pocket_shards)

        node_features_list, node_pos_list = [], []
        frag_embeds_list, ref_coords_list = [], []

        for idx, shard in enumerate(all_shards):
            node_type = [0.0, 1.0] if idx < n_pocket else [1.0, 0.0]
            other_features = compute_normalized_features(shard)
            node_features_list.append(node_type + other_features)

            rel_centroid = np.array(shard['centroid_3d']) - meb_center
            node_pos_list.append(rel_centroid.tolist())
            rel_ref = np.array(shard['ref_coord_3d']) - meb_center
            ref_coords_list.append(rel_ref.tolist())

            frag_embeds_list.append(get_shard_embedding(shard))

        x = torch.tensor(node_features_list, dtype=torch.float)
        pos = torch.tensor(node_pos_list, dtype=torch.float)
        frag_embeds = torch.tensor(frag_embeds_list, dtype=torch.float)
        ref_coords = torch.tensor(np.array(ref_coords_list), dtype=torch.float)

        edge_list, edge_attr_list = [], []
        for (u, v), e_type in all_edge_attrs.items():
            edge_list.append([u, v])
            attr = [0.0]*5; attr[e_type] = 1.0
            edge_attr_list.append(attr)

        if edge_list:
            e_idx = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
            e_attr = torch.tensor(edge_attr_list, dtype=torch.float)
            e_idx = torch.cat([e_idx, e_idx.flip(0)], dim=1)
            e_attr = torch.cat([e_attr, e_attr], dim=0)
        else:
            e_idx = torch.empty((2, 0), dtype=torch.long)
            e_attr = torch.empty((0, 5), dtype=torch.float)

        data = Data(
            x=x, pos=pos, edge_index=e_idx, edge_attr=e_attr,
            pdb_id=complex_id, frag_embeds=frag_embeds, ref_coords=ref_coords,
            meb_center=torch.tensor(meb_center, dtype=torch.float),
            n_pocket_nodes=n_pocket
        )
        return data

    # =================================================================
    # 内部辅助函数: 组装仅口袋的 PyG Data (Creative 模式: 只有口袋)
    # =================================================================
    def assemble_pocket_only_pyg_data(pocket_shards, pocket_edge_attrs,
                                         meb_center, complex_id):
        # --- 过滤无效 shard ---
        valid_pocket = [s for s in pocket_shards if s.get('centroid_3d') is not None and s.get('ref_coord_3d') is not None]
        if len(valid_pocket) < len(pocket_shards):
            print(f"  [警告] 过滤无3D坐标口袋碎片: -{len(pocket_shards)-len(valid_pocket)}")
        old_to_new = {}
        for new_idx, shard in enumerate(valid_pocket):
            old_to_new[shard['Shard ID']] = new_idx
        pocket_edge_attrs = {(old_to_new[u], old_to_new[v]): t for (u,v), t in pocket_edge_attrs.items() if u in old_to_new and v in old_to_new}
        pocket_shards = valid_pocket
        # --- 过滤结束 ---

        node_features_list, node_pos_list = [], []
        frag_embeds_list, ref_coords_list = [], []

        for shard in pocket_shards:
            node_type = [0.0, 1.0]
            other_features = compute_normalized_features(shard)
            node_features_list.append(node_type + other_features)

            rel_centroid = np.array(shard['centroid_3d']) - meb_center
            node_pos_list.append(rel_centroid.tolist())
            rel_ref = np.array(shard['ref_coord_3d']) - meb_center
            ref_coords_list.append(rel_ref.tolist())

            frag_embeds_list.append(get_shard_embedding(shard))

        x = torch.tensor(node_features_list, dtype=torch.float)
        pos = torch.tensor(node_pos_list, dtype=torch.float)
        frag_embeds = torch.tensor(frag_embeds_list, dtype=torch.float)
        ref_coords = torch.tensor(np.array(ref_coords_list), dtype=torch.float)

        edge_list, edge_attr_list = [], []
        for (u, v), e_type in pocket_edge_attrs.items():
            edge_list.append([u, v])
            attr = [0.0]*5; attr[e_type] = 1.0
            edge_attr_list.append(attr)

        if edge_list:
            e_idx = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
            e_attr = torch.tensor(edge_attr_list, dtype=torch.float)
            e_idx = torch.cat([e_idx, e_idx.flip(0)], dim=1)
            e_attr = torch.cat([e_attr, e_attr], dim=0)
        else:
            e_idx = torch.empty((2, 0), dtype=torch.long)
            e_attr = torch.empty((0, 5), dtype=torch.float)

        data = Data(
            x=x, pos=pos, edge_index=e_idx, edge_attr=e_attr,
            pdb_id=complex_id, frag_embeds=frag_embeds, ref_coords=ref_coords,
            meb_center=torch.tensor(meb_center, dtype=torch.float)
        )
        return data

    # =================================================================
    # 辅助函数: 口袋碎片化 + 边构建 (两种模式共用)
    # =================================================================
    def fragmentize_pocket(pocket_mol, complex_id):
        """对口袋分子进行碎片化处理，返回 (pocket_shards, total_dropped)"""
        for atom in pocket_mol.GetAtoms():
            atom.SetIntProp("_original_index", atom.GetIdx())

        raw_pocket_frags = Chem.GetMolFrags(pocket_mol, asMols=True, sanitizeFrags=False)
        pocket_fragments_no_water = []
        for f in raw_pocket_frags:
            is_water = False
            if f.GetNumAtoms() == 1:
                atom = f.GetAtomWithIdx(0)
                if atom.GetAtomicNum() == 8 and atom.GetFormalCharge() == 0:
                    is_water = True
            if not is_water:
                pocket_fragments_no_water.append(f)

        pocket_shards = []
        p_offset = 0
        total_dropped = 0
        for raw_frag in pocket_fragments_no_water:
            res, _, d_cnt = process_single_fragment(raw_frag, 'pocket', complex_id, p_offset)
            total_dropped += d_cnt
            if res:
                pocket_shards.extend(res)
                p_offset += len(res)

        return pocket_shards, total_dropped

    def fragmentize_ligand_and_compute_center(ligand_mol, complex_id, id_start_offset=0):
        """对配体进行碎片化处理，并基于全部 ligand shard centroid 计算 meb_center"""
        if ligand_mol is None:
            return [], None, 1

        for atom in ligand_mol.GetAtoms():
            atom.SetIntProp("_original_index", atom.GetIdx())

        ligand_connected_components = Chem.GetMolFrags(ligand_mol, asMols=True, sanitizeFrags=False)
        ligand_shards = []
        l_offset = id_start_offset
        total_dropped = 0

        for raw_frag in ligand_connected_components:
            res, err, d_cnt = process_single_fragment(raw_frag, 'ligand', complex_id, l_offset)
            if err:
                print(f"    [配体碎片化警告] {err}")
            if d_cnt > 0:
                print(f"    [配体碎片化] 丢弃了 {d_cnt} 个片段")
            if res:
                for s in res:
                    print(f"    [配体碎片] Shard {s['Shard ID']}: {s.get('smiles','?')} (origin={s['origin']})")
                ligand_shards.extend(res)
                l_offset += len(res)
            total_dropped += d_cnt

        if not ligand_shards:
            return [], None, total_dropped

        ligand_centroids = []
        for shard in ligand_shards:
            centroid = shard.get('centroid_3d')
            if centroid is None:
                raise ValueError('配体 shard 缺失 centroid_3d，无法按全量节点计算 meb_center')
            ligand_centroids.append(np.array(centroid, dtype=float))

        ligand_centroids = np.array(ligand_centroids, dtype=float)
        meb_center, _ = utils.calculate_minimum_enclosing_ball(ligand_centroids)
        return ligand_shards, meb_center, total_dropped

    def build_pocket_edges(pocket_shards, pocket_mol):
        """构建口袋节点之间的边"""
        pocket_edges = set()
        pocket_edge_attrs = {}
        atom_to_shard_map = {}
        for s in pocket_shards:
            if 'mol' in s:
                for atom in s['mol'].GetAtoms():
                    if atom.HasProp("_original_index"):
                        atom_to_shard_map[atom.GetIntProp("_original_index")] = s['Shard ID']

        mol_for_edges = Chem.RemoveHs(pocket_mol)
        for bond in mol_for_edges.GetBonds():
            idx1, idx2 = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
            sid1 = atom_to_shard_map.get(idx1)
            sid2 = atom_to_shard_map.get(idx2)
            if sid1 is not None and sid2 is not None and sid1 != sid2:
                et = tuple(sorted((sid1, sid2)))
                if et not in pocket_edges:
                    pocket_edges.add(et)
                    pocket_edge_attrs[et] = 1

        for s1, s2 in combinations(pocket_shards, 2):
            sid1, sid2 = s1['Shard ID'], s2['Shard ID']
            et = tuple(sorted((sid1, sid2)))
            if et in pocket_edges:
                continue
            if s1.get('centroid_3d') and s2.get('centroid_3d'):
                if scipy.spatial.distance.euclidean(s1['centroid_3d'], s2['centroid_3d']) < 12.0:
                    c1 = utils.get_atom_coordinates(s1.get('clean_mol'))
                    c2 = utils.get_atom_coordinates(s2.get('clean_mol'))
                    if c1.size > 0 and c2.size > 0:
                        if np.min(scipy.spatial.distance.cdist(c1, c2)) < 6.0:
                            pocket_edges.add(et)
                            pocket_edge_attrs[et] = 4

        return pocket_edge_attrs
    
    def build_all_edges(pocket_shards, ligand_shards, pocket_mol, ligand_mol):                                                      
      """                                                                                                                               构建完整的 5 种边类型，与训练数据管线一致。                                                                                 
      Type 0: LLCov  (配体-配体共价)                                                                                              
      Type 1: PPCov  (口袋-口袋共价)
      Type 2: LPGlob (配体-口袋空间)
      Type 3: LLGlob (配体-配体空间)
      Type 4: PPGlob (口袋-口袋空间)
      """
      all_shards = pocket_shards + ligand_shards
      edges = set()
      edge_attrs = {}

      # --- Step 1: PP 共价边 (Type 1) ---
      pocket_atom_map = {}
      for s in pocket_shards:
          if 'mol' in s:
              for atom in s['mol'].GetAtoms():
                  if atom.GetAtomicNum() != 0 and atom.HasProp("_original_index"):
                      pocket_atom_map[atom.GetIntProp("_original_index")] = s['Shard ID']

      mol_pp = Chem.RemoveHs(pocket_mol)
      for bond in mol_pp.GetBonds():
          sid1 = pocket_atom_map.get(bond.GetBeginAtomIdx())
          sid2 = pocket_atom_map.get(bond.GetEndAtomIdx())
          if sid1 is not None and sid2 is not None and sid1 != sid2:
              et = tuple(sorted((sid1, sid2)))
              if et not in edges:
                  edges.add(et)
                  edge_attrs[et] = 1

      # --- Step 2: LL 共价边 (Type 0) ---
      ligand_atom_map = {}
      for s in ligand_shards:
          if 'mol' in s:
              for atom in s['mol'].GetAtoms():
                  if atom.GetAtomicNum() != 0 and atom.HasProp("_original_index"):
                      ligand_atom_map[atom.GetIntProp("_original_index")] = s['Shard ID']

      mol_ll = Chem.RemoveHs(ligand_mol)
      for bond in mol_ll.GetBonds():
          sid1 = ligand_atom_map.get(bond.GetBeginAtomIdx())
          sid2 = ligand_atom_map.get(bond.GetEndAtomIdx())
          if sid1 is not None and sid2 is not None and sid1 != sid2:
              et = tuple(sorted((sid1, sid2)))
              if et not in edges:
                  edges.add(et)
                  edge_attrs[et] = 0

      # --- Step 3: 空间接近边 (Type 2/3/4) ---
      for s1, s2 in combinations(all_shards, 2):
          et = tuple(sorted((s1['Shard ID'], s2['Shard ID'])))
          if et in edges:
              continue
          if s1.get('centroid_3d') and s2.get('centroid_3d'):
              if scipy.spatial.distance.euclidean(s1['centroid_3d'], s2['centroid_3d']) < 12.0:
                  c1 = utils.get_atom_coordinates(s1.get('clean_mol'))
                  c2 = utils.get_atom_coordinates(s2.get('clean_mol'))
                  if c1.size > 0 and c2.size > 0:
                      if np.min(scipy.spatial.distance.cdist(c1, c2)) < 6.0:
                          edges.add(et)
                          origins = sorted([s1['origin'], s2['origin']])
                          if origins == ['ligand', 'pocket']:
                              edge_attrs[et] = 2   # LPGlob
                          elif origins == ['ligand', 'ligand']:
                              edge_attrs[et] = 3   # LLGlob
                          else:
                              edge_attrs[et] = 4   # PPGlob

      return edge_attrs
    # =================================================================
    # 主处理循环 (逐个复合物)
    # =================================================================
    success_count_a = 0
    fail_count_a = 0

    for file_idx, (complex_id, pdb_path) in enumerate(zip(task_ids, task_pdb_files)):
        print(f"\n{'='*60}")
        print(f"[{file_idx+1}/{len(task_ids)}] [{TASK_MODE}] 处理: {complex_id}")
        print(f"{'='*60}")

        try:
            # =============================================================
            # ★ TASK_B 模式: 无参考配体流程
            # =============================================================
            if TASK_MODE == "TASK_B":

                # --- 阶段 B0: 读取纯蛋白 ---
                protein_mol, protein_lines = clean_protein_from_pdb(pdb_path)
                if protein_mol is None:
                    raise ValueError("蛋白分子加载失败")
                print(f"  [TASK_B] 纯蛋白: {protein_mol.GetNumAtoms()} 原子 (已去除所有 HETATM)")

                # --- 阶段 B1: 计算蛋白 MEB 中心 ---
                conf = protein_mol.GetConformer()
                all_coords = np.array([list(conf.GetAtomPosition(i)) for i in range(protein_mol.GetNumAtoms())])
                initial_center, _ = utils.calculate_minimum_enclosing_ball(all_coords)
                print(f"  [TASK_B] 蛋白 MEB 中心: ({initial_center[0]:.3f}, {initial_center[1]:.3f}, {initial_center[2]:.3f})")

                # --- 阶段 B2: 可视化 + 坐标调试 (终极 IFrame 隔离版) ---
                from IPython.display import HTML, display, clear_output
                import urllib.parse
                import base64
                
                coord_result = {'center': initial_center.copy(), 'confirmed': False}

                x_input = widgets.FloatText(value=round(float(initial_center[0]), 3), description='X:', layout=widgets.Layout(width='200px'))
                y_input = widgets.FloatText(value=round(float(initial_center[1]), 3), description='Y:', layout=widgets.Layout(width='200px'))
                z_input = widgets.FloatText(value=round(float(initial_center[2]), 3), description='Z:', layout=widgets.Layout(width='200px'))

                debug_btn = widgets.Button(description='调试预览', button_style='warning', icon='eye')
                confirm_coord_btn = widgets.Button(description='确认坐标', button_style='success', icon='check')
                skip_btn_b = widgets.Button(description='跳过此蛋白', button_style='warning', icon='close')
                
                # 我们准备两个单独的 Output，避免互相干扰
                view_output = widgets.Output() 
                status_output = widgets.Output()

                # 读取 PDB 数据并进行 URL 编码，防止其内部换行符破坏 HTML 结构
                with open(pdb_path, 'r') as f:
                    pdb_data = f.read()
                safe_pdb_data = urllib.parse.quote(pdb_data) 

                def render_iframe_view(cx, cy, cz):
                    """生成一个完全独立的 HTML 页面，并打包进 iframe 中"""
                    html_content = f"""
                    <!DOCTYPE html>
                    <html>
                    <head>
                        <script src="https://3dmol.org/build/3Dmol-min.js"></script>
                    </head>
                    <body style="margin:0; padding:0; overflow:hidden; background-color:white;">
                        <div id="container" style="width: 100%; height: 95vh; position: relative;"></div>
                        <script>
                            let element = document.getElementById('container');
                            let viewer = $3Dmol.createViewer(element, {{backgroundColor: 'white'}});
                            let pdbData = decodeURIComponent("{safe_pdb_data}");
                            
                            viewer.addModel(pdbData, "pdb");
                            viewer.setStyle({{hetflag: false}}, {{cartoon: {{color: 'lightgray', opacity: 0.5}}}});
                            viewer.setStyle({{hetflag: true}}, {{}});
                            
                            // 红色球体，透明度 0.5
                            viewer.addSphere({{
                                center: {{x: {cx}, y: {cy}, z: {cz}}}, 
                                radius: 16.0, 
                                color: 'orange', 
                                opacity: 0.5
                            }});
                            viewer.addLabel("Center ({cx:.1f}, {cy:.1f}, {cz:.1f})", {{
                                position: {{x: {cx}, y: {cy}, z: {cz}}},
                                backgroundColor: 'red', fontColor: 'white',
                                fontSize: 12, backgroundOpacity: 1.0
                            }});
                            
                            viewer.zoomTo();
                            viewer.render();
                        </script>
                    </body>
                    </html>
                    """

                    
                    # 对 HTML 内容进行实体转义，安全地放入属性中
                    escaped_html = html.escape(html_content)
                    iframe_html = f'<div><iframe srcdoc="{escaped_html}" width="100%" height="500px" style="border:none;"></iframe></div>'
                    return HTML(iframe_html)

                # --- 初始渲染 ---
                with view_output:
                    display(render_iframe_view(*initial_center))

                def on_debug(btn):
                    cx, cy, cz = x_input.value, y_input.value, z_input.value
                    coord_result['center'] = np.array([cx, cy, cz])
                    
                    # 每次点击，在专属 Output 内抹掉旧沙盒，挂载新沙盒
                    with view_output:
                        clear_output(wait=True)
                        display(render_iframe_view(cx, cy, cz))
                        
                    with status_output:
                        clear_output()
                        print(f"�� 画面已强制刷新，预览坐标: ({cx:.3f}, {cy:.3f}, {cz:.3f})")

                def on_confirm_coord(btn):
                    if coord_result['confirmed']:
                        return
                    cx, cy, cz = x_input.value, y_input.value, z_input.value
                    coord_result['center'] = np.array([cx, cy, cz])
                    coord_result['confirmed'] = True
                    x_input.disabled = True; y_input.disabled = True; z_input.disabled = True
                    debug_btn.disabled = True; confirm_coord_btn.disabled = True; skip_btn_b.disabled = True
                    with status_output:
                        clear_output()
                        print(f"✅ 已确认坐标: ({cx:.3f}, {cy:.3f}, {cz:.3f})")

                def on_skip_b(btn):
                    if coord_result['confirmed']:
                        return
                    coord_result['center'] = None
                    coord_result['confirmed'] = True
                    x_input.disabled = True; y_input.disabled = True; z_input.disabled = True
                    debug_btn.disabled = True; confirm_coord_btn.disabled = True; skip_btn_b.disabled = True
                    with status_output:
                        clear_output()
                        print(f"⏭️ 已跳过 {complex_id}")

                debug_btn.on_click(on_debug)
                confirm_coord_btn.on_click(on_confirm_coord)
                skip_btn_b.on_click(on_skip_b)

                print("  请调整球心坐标，点击[调试预览]查看效果，满意后点击[确认坐标]:")
                
                # 依次显示组件
                display(view_output)
                display(widgets.HBox([x_input, y_input, z_input]))
                display(widgets.HBox([debug_btn, confirm_coord_btn, skip_btn_b]))
                display(status_output)

                with ui_events() as poll:
                    while not coord_result['confirmed']:
                        poll(10)
                        time.sleep(0.1)

                if coord_result['center'] is None:
                    fail_count_a += 1
                    continue

                user_center = coord_result['center']
                meb_center = user_center

                # --- 阶段 B3: 以确认坐标为中心 12Å 截取局部口袋 ---
                dummy_ligand_coords = np.array([user_center])
                pocket_mol, pocket_pdb_lines = utils.extract_pocket_around_ligand(
                    pdb_path, dummy_ligand_coords, protein_lines, radius=16.0
                )

                if pocket_mol is None:
                    raise ValueError("口袋截取失败")

                print(f"  [TASK_B] 截取口袋: {pocket_mol.GetNumAtoms()} 原子 (以确认坐标为中心, 12Å)")

                # --- 阶段 B4: 进入 Creative 模式后半段 ---
                pocket_shards, total_dropped = fragmentize_pocket(pocket_mol, complex_id)

                if total_dropped > 10:
                    raise ValueError(f"质量控制失败: 丢失 {total_dropped} 个片段")
                if not pocket_shards:
                    raise ValueError("未能生成有效口袋碎片")

                print(f"  [TASK_B] 碎片化完成: 口袋 {len(pocket_shards)} 节点")

                pocket_edge_attrs = build_pocket_edges(pocket_shards, pocket_mol)

                pocket_data = assemble_pocket_only_pyg_data(
                    pocket_shards, pocket_edge_attrs, meb_center, complex_id
                )

                print(f"  [TASK_B] PyG Data: {pocket_data.x.size(0)} 节点, {pocket_data.edge_index.size(1)} 边")

                # 保存到 TASK_B_OUTPUT_DIR
                out_dir = os.path.join(TASK_B_OUTPUT_DIR, complex_id)
                os.makedirs(out_dir, exist_ok=True)
                torch.save(pocket_data, os.path.join(out_dir, f"{complex_id}.pt"))

                centered_pdb_lines = []
                for line in pocket_pdb_lines:
                    try:
                        x = float(line[30:38]) - meb_center[0]
                        y = float(line[38:46]) - meb_center[1]
                        z = float(line[46:54]) - meb_center[2]
                        new_line = f"{line[:30]}{x:8.3f}{y:8.3f}{z:8.3f}{line[54:]}"
                        centered_pdb_lines.append(new_line)
                    except:
                        centered_pdb_lines.append(line)

                with open(os.path.join(out_dir, 'pocket.pdb'), 'w') as f:
                    f.writelines(centered_pdb_lines)
                    f.write('END\n')

                print(f"  [TASK_B] ✅ 已保存至: {out_dir}/")
                success_count_a += 1

            # =============================================================
            # ★ TASK_A 模式: 有参考配体流程 (原有逻辑完整保留)
            # =============================================================
            elif TASK_MODE == "TASK_A":

                # =========================================================
                # 阶段 0: 交互式选择处理模式 (Survival / Creative)
                # =========================================================
                mode_output = widgets.Output()
                mode_result = {'mode': None, 'confirmed': False}

                mode_selector = widgets.RadioButtons(
                    options=['Survival', 'Creative'],
                    value='Survival',
                    description='处理模式:',
                    style={'description_width': 'initial'},
                    layout=widgets.Layout(width='300px')
                )

                mode_info = widgets.HTML(value="""
                <div style="padding:10px; background:#f0f0f0; border-radius:5px; margin:5px 0;">
                <b>Survival 模式:</b> 保留配体，口袋+配体一起碎片化，支持交互式删除节点，保存到 TASK_A<br>
                <b>Creative 模式:</b> 选择配体后计算 MEB 中心，然后删除所有配体，仅处理口袋蛋白，保存到 TASK_B
                </div>
                """)

                mode_btn = widgets.Button(description='确认模式', button_style='primary', icon='check')

                def on_mode_confirm(btn):
                    if mode_result['confirmed']:
                        return
                    mode_selector.disabled = True
                    mode_btn.disabled = True
                    mode_btn.description = '已确认'
                    mode_result['mode'] = mode_selector.value
                    mode_result['confirmed'] = True
                    with mode_output:
                        clear_output()
                        print(f"✅ 已选择 {mode_result['mode']} 模式")

                mode_btn.on_click(on_mode_confirm)

                print("  请选择处理模式:")
                display(mode_info)
                display(widgets.HBox([mode_selector, mode_btn]))
                display(mode_output)
                with ui_events() as poll:
                    while not mode_result['confirmed']:
                        poll(10)
                        time.sleep(0.1)
                current_mode = mode_result['mode']

                # =========================================================
                # 阶段 1: 解析 PDB，检测配体候选，清理干扰项
                # =========================================================
                ligand_candidates, protein_lines = utils.parse_hetatm_residues(pdb_path)

                # 【新增】清理干扰项 (水分子、离子、酸根等，保留金属元素)
                ligand_candidates, removed_items = clean_interference_from_candidates(ligand_candidates)

                if removed_items:
                    print(f"  已清理 {len(removed_items)} 个干扰项 (水/离子/酸根/缓冲液等):")
                    for item in removed_items:
                        print(f"    - {item['label']}")

                if not ligand_candidates:
                    print("  [跳过] 清理后未检测到有效配体候选")
                    fail_count_a += 1
                    continue

                print(f"  检测到 {len(ligand_candidates)} 个有效配体候选:")
                for i, cand in enumerate(ligand_candidates):
                    print(f"    [{i}] {cand['label']}")

                # =========================================================
                # 阶段 2: 交互式选择配体 (py3Dmol + ipywidgets)
                # =========================================================
                output_area = widgets.Output()
                result_container = {'selected': None, 'confirmed': False}

                with open(pdb_path, 'r') as f:
                    pdb_data = f.read()
                    view = py3Dmol.view(width='100%', height=500)
                    view.addModel(pdb_data, 'pdb')
                    view.setStyle({}, {})
                    view.addStyle({'hetflag': False}, {'cartoon': {'color': 'lightgray', 'opacity': 0.5}})
                    view.addStyle({'hetflag': True}, {'stick': {}})

                    for cand in ligand_candidates:
                        if 'coords' in cand and len(cand['coords']) > 0:
                            center = np.mean(cand['coords'], axis=0)
                            view.addLabel(cand['label'], {
                                'position': {'x': center[0], 'y': center[1], 'z': center[2]},
                                'backgroundColor': 'black',
                                'fontColor': 'white',
                                'fontSize': 13,
                                'backgroundOpacity': 0.7,
                                'alignment': 'center'
                            })
                    view.zoomTo()

                options_dict = {cand['label']: i for i, cand in enumerate(ligand_candidates)}
                dropdown = widgets.Dropdown(
                    options=options_dict,
                    description='选择配体:',
                    style={'description_width': 'initial'}
                )

                confirm_btn = widgets.Button(description='确认选择', button_style='success', icon='check')
                skip_btn = widgets.Button(description='跳过此复合物', button_style='warning', icon='close')

                def on_confirm(btn):
                    if result_container['confirmed']:
                        return
                    dropdown.disabled = True
                    confirm_btn.disabled = True
                    skip_btn.disabled = True
                    confirm_btn.description = '已确认，处理中...'
                    result_container['selected'] = dropdown.value
                    result_container['confirmed'] = True
                    with output_area:
                        clear_output()
                        sel = ligand_candidates[dropdown.value]
                        print(f"✅ 成功确认选择配体: {sel['label']}")
                        if current_mode == 'Creative':
                            print("⏳ [Creative 模式] 正在计算 MEB 中心，删除配体并处理口袋...")
                        else:
                            print("⏳ [Survival 模式] 正在提取配体并截取其周围口袋，请稍候...")

                def on_skip(btn):
                    if result_container['confirmed']:
                        return
                    dropdown.disabled = True
                    confirm_btn.disabled = True
                    skip_btn.disabled = True
                    skip_btn.description = '已跳过'
                    result_container['confirmed'] = True
                    with output_area:
                        clear_output()
                        print(f"⏭️ 已跳过 {complex_id}")

                confirm_btn.on_click(on_confirm)
                skip_btn.on_click(on_skip)

                view.show()
                display(widgets.HBox([dropdown, confirm_btn, skip_btn]))
                display(output_area)

                print("  请在上方选择配体并点击 [确认选择]...")
                with ui_events() as poll:
                    while not result_container['confirmed']:
                        poll(10)
                        time.sleep(0.1)

                if result_container['selected'] is None:
                    fail_count_a += 1
                    continue

                selected_ligand = ligand_candidates[result_container['selected']]

                # =========================================================
                # 根据模式分流处理
                # =========================================================

                if current_mode == 'Creative':

                    # --- 阶段 C1: 先对配体进行碎片化，并基于全部 ligand shard centroid 计算 MEB 中心 ---
                    ligand_mol = utils.build_ligand_mol_from_pdb_lines(selected_ligand['atom_lines'])
                    if ligand_mol is None:
                        raise ValueError("配体分子加载失败")
                    ligand_shards, meb_center, ligand_dropped = fragmentize_ligand_and_compute_center(
                        ligand_mol, complex_id, 0
                    )
                    if ligand_dropped > 10:
                        raise ValueError(f"质量控制失败: 丢失 {ligand_dropped} 个片段")
                    if not ligand_shards or meb_center is None:
                        raise ValueError("未能生成有效配体碎片或 MEB 中心")
                    print(f"  [Creative] 配体 shard MEB 中心: ({meb_center[0]:.3f}, {meb_center[1]:.3f}, {meb_center[2]:.3f})")

                    # --- 阶段 C2: 截取口袋 (12 angstrom，基于选中配体的坐标) ---
                    pocket_mol, pocket_pdb_lines = utils.extract_pocket_around_ligand(
                        pdb_path, selected_ligand['coords'], protein_lines, radius=12.0
                    )

                    if pocket_mol is None:
                        raise ValueError("口袋分子加载失败")

                    print(f"  [Creative] 口袋: {pocket_mol.GetNumAtoms()} 原子 (配体已删除，不参与后续处理)")

                    # --- 阶段 C3: 口袋碎片化 ---
                    pocket_shards, total_dropped = fragmentize_pocket(pocket_mol, complex_id)

                    if total_dropped > 10:
                        raise ValueError(f"质量控制失败: 丢失 {total_dropped} 个片段")

                    if not pocket_shards:
                        raise ValueError("未能生成有效口袋碎片")

                    print(f"  [Creative] 碎片化完成: 口袋 {len(pocket_shards)} 节点")

                    # --- 阶段 C4: 构建口袋边 ---
                    pocket_edge_attrs = build_pocket_edges(pocket_shards, pocket_mol)

                    # --- 阶段 C5: 组装仅口袋的 PyG Data ---
                    pocket_data = assemble_pocket_only_pyg_data(
                        pocket_shards, pocket_edge_attrs, meb_center, complex_id
                    )

                    print(f"  [Creative] PyG Data: {pocket_data.x.size(0)} 节点, {pocket_data.edge_index.size(1)} 边")

                    # --- 阶段 C6: 保存到 TASK_B ---
                    out_dir = os.path.join(TASK_B_OUTPUT_DIR, complex_id)
                    os.makedirs(out_dir, exist_ok=True)
                    torch.save(pocket_data, os.path.join(out_dir, f"{complex_id}.pt"))

                    # 保存归中后的口袋 PDB
                    centered_pdb_lines = []
                    for line in pocket_pdb_lines:
                        try:
                            x = float(line[30:38]) - meb_center[0]
                            y = float(line[38:46]) - meb_center[1]
                            z = float(line[46:54]) - meb_center[2]
                            new_line = f"{line[:30]}{x:8.3f}{y:8.3f}{z:8.3f}{line[54:]}"
                            centered_pdb_lines.append(new_line)
                        except:
                            centered_pdb_lines.append(line)

                    with open(os.path.join(out_dir, 'pocket.pdb'), 'w') as f:
                        f.writelines(centered_pdb_lines)
                        f.write('END\n')

                    _ligand_mol = utils.build_ligand_mol_from_pdb_lines(selected_ligand['atom_lines'])
                    if _ligand_mol is not None:
                        norm_ligand = Chem.Mol(_ligand_mol) # 建立副本
                        conf = norm_ligand.GetConformer()
                        for i in range(norm_ligand.GetNumAtoms()):
                            pos = np.array(conf.GetAtomPosition(i))
                            new_pos = pos - meb_center
                            conf.SetAtomPosition(i, (float(new_pos[0]), float(new_pos[1]), float(new_pos[2])))
                        
                        sdf_out_path = os.path.join(out_dir, f"{complex_id}_ligand_normalized.sdf")
                        with Chem.SDWriter(sdf_out_path) as writer:
                            writer.write(norm_ligand)
                        print(f"  [Creative] ✅ 已保存归一化配体: {sdf_out_path}")

                    _ligand_mol = utils.build_ligand_mol_from_pdb_lines(selected_ligand['atom_lines'])
                    if _ligand_mol is not None:
                        norm_ligand = Chem.Mol(_ligand_mol) # 建立副本
                        conf = norm_ligand.GetConformer()
                        for i in range(norm_ligand.GetNumAtoms()):
                            pos = np.array(conf.GetAtomPosition(i))
                            new_pos = pos - meb_center
                            conf.SetAtomPosition(i, (float(new_pos[0]), float(new_pos[1]), float(new_pos[2])))
                        
                        sdf_out_path = os.path.join(out_dir, f"{complex_id}_ligand_normalized.sdf")
                        with Chem.SDWriter(sdf_out_path) as writer:
                            writer.write(norm_ligand)
                        print(f"  [Creative] ✅ 已保存归一化配体: {sdf_out_path}")

                    # ==========================================
                    # 【新增】处理并保存完整的归一化蛋白文件
                    # ==========================================
                    centered_full_pdb_lines = []
                    for line in protein_lines:
                        try:
                            # 按照 PDB 固定列宽截取坐标并平移
                            x = float(line[30:38]) - meb_center[0]
                            y = float(line[38:46]) - meb_center[1]
                            z = float(line[46:54]) - meb_center[2]
                            new_line = f"{line[:30]}{x:8.3f}{y:8.3f}{z:8.3f}{line[54:]}"
                            centered_full_pdb_lines.append(new_line)
                        except:
                            # 如果该行格式异常（非标准 ATOM 行），原样保留
                            centered_full_pdb_lines.append(line)

                    full_protein_path = os.path.join(out_dir, 'full_protein_normalized.pdb')
                    with open(full_protein_path, 'w') as f:
                        f.writelines(centered_full_pdb_lines)
                        f.write('END\n')
                        
                    print(f"  [Creative] ✅ 已保存归一化完整受体: {full_protein_path}")
                    # ==========================================
                    # 新增代码结束
                    # ==========================================

                    print(f"  [Creative] ✅ 已保存至: {out_dir}/")
                    success_count_a += 1

                else:
                    # =============================================================
                    # ★ Survival 模式流程 (保持原逻辑)
                    # =============================================================

                    # --- 阶段 S3: 截取口袋 (12 angstrom) + 加载配体 ---
                    pocket_mol, pocket_pdb_lines = utils.extract_pocket_around_ligand(
                        pdb_path, selected_ligand['coords'], protein_lines, radius=12.0
                    )
                    ligand_mol = utils.build_ligand_mol_from_pdb_lines(selected_ligand['atom_lines'])

                    if pocket_mol is None or ligand_mol is None:
                        raise ValueError("口袋或配体分子加载失败")

                    print(f"  [Survival] 口袋: {pocket_mol.GetNumAtoms()} 原子 | 配体: {ligand_mol.GetNumAtoms()} 原子")

                    # --- 阶段 S4: 碎片化处理 ---
                    # 口袋碎片化
                    pocket_shards, total_dropped = fragmentize_pocket(pocket_mol, complex_id)

                    # 配体碎片化 + 基于全部 ligand shard centroid 计算 MEB 中心
                    ligand_shards, meb_center, ligand_dropped = fragmentize_ligand_and_compute_center(
                        ligand_mol, complex_id, len(pocket_shards)
                    )
                    total_dropped += ligand_dropped

                    if total_dropped > 10:
                        raise ValueError(f"质量控制失败: 丢失 {total_dropped} 个片段")

                    if not pocket_shards or not ligand_shards or meb_center is None:
                        raise ValueError("未能生成有效碎片")

                    print(f"  [Survival] 碎片化完成: 口袋 {len(pocket_shards)} 节点 | 配体 {len(ligand_shards)} 节点")

                    # --- 阶段 S5: 边构建 + MEB + 组装 PyG Data ---
                    all_edge_attrs = build_all_edges(pocket_shards, ligand_shards, pocket_mol, ligand_mol)

                    print(f"  [Survival] 配体 shard MEB 中心: ({meb_center[0]:.3f}, {meb_center[1]:.3f}, {meb_center[2]:.3f})")

                    # 组装 PyG Data
                    combined_data = assemble_combined_pyg_data(
                        pocket_shards, ligand_shards, all_edge_attrs, meb_center, complex_id
                    )

                    print(f"  [Survival] PyG Data: {combined_data.x.size(0)} 节点, {combined_data.edge_index.size(1)} 边")

                    # --- 阶段 S6: Kabsch 反映射 + 碎片可视化 + 交互式删除 ---
                    all_shards = pocket_shards + ligand_shards
                    n_pocket = len(pocket_shards)

                    # 只对配体碎片进行 Kabsch 反映射和 3D 渲染
                    ligand_aligned_mols = []
                    for shard in ligand_shards:
                        if shard.get('centroid_3d') and shard.get('ref_coord_3d') is not None:
                            smi = shard.get('smiles', '<UNK>')
                            mol_3d = utils.kabsch_align_fragment_to_reference(
                                smi,
                                np.array(shard['centroid_3d']),
                                np.array(shard['ref_coord_3d'])
                            )
                            ligand_aligned_mols.append(mol_3d)
                        else:
                            ligand_aligned_mols.append(None)

                    # 元素颜色映射 (CPK 配色方案)
                    ELEMENT_COLORS = {
                        6: "#FFFF0F",   # C
                        7: '#3050F8',   # N
                        8: '#FF0D0D',   # O
                        9: '#90E050',   # F
                        15: "#E5D7CA",  # P
                        16: "#FF9D0B",  # S
                        17: '#1FF01F',  # Cl
                        35: '#A62929',  # Br
                        53: '#940094',  # I
                    }

                    frag_view = py3Dmol.view(width='100%', height=500)

                    for lig_idx, (mol_3d, shard) in enumerate(zip(ligand_aligned_mols, ligand_shards)):
                        if mol_3d is None:
                            continue
                        try:
                            sdf_block = Chem.MolToMolBlock(mol_3d)
                            frag_view.addModel(sdf_block, 'sdf')

                            if mol_3d.GetNumAtoms() == 1:
                                atom = mol_3d.GetAtomWithIdx(0)
                                color = ELEMENT_COLORS.get(atom.GetAtomicNum(), '#FF6644')
                                frag_view.setStyle({'model': -1}, {'sphere': {'color': color, 'radius': 0.2}})
                            else:
                                frag_view.setStyle({'model': -1}, {'stick': {'colorscheme': 'Jmol', 'radius': 0.2}})

                            # 添加标签 (编号与复选框一致)
                            real_idx = n_pocket + lig_idx  # all_shards 中的真实索引
                            smi = shard.get('smiles', '?')[:25]
                            label_text = f"[{real_idx}] {smi}"
                            conf = mol_3d.GetConformer(0)
                            coords = [list(conf.GetAtomPosition(a)) for a in range(mol_3d.GetNumAtoms())]
                            cx, cy, cz = np.mean(coords, axis=0)
                            frag_view.addLabel(label_text, {
                                'position': {'x': float(cx), 'y': float(cy), 'z': float(cz)},
                                'backgroundColor': 'black',
                                'fontColor': 'white',
                                'fontSize': 12,
                                'backgroundOpacity': 0.7,
                                'alignment': 'center'
                            })
                        except:
                            pass
                    frag_view.zoomTo()

                    # 复选框
                    delete_output = widgets.Output()
                    delete_result = {'deleted': set(), 'confirmed': False}

                    checkboxes = []
                    checkbox_index_map = {}  # checkbox列表位置 -> all_shards中的真实索引
                    for i, shard in enumerate(all_shards):
                        if i < n_pocket:
                            continue  # 跳过口袋节点，不生成复选框
                        smi = shard.get('smiles', '?')[:35]
                        cb = widgets.Checkbox(
                            value=False,
                            description=f"[{i}] [L] {smi}",
                            indent=False,
                            style={'description_width': 'initial'},
                            layout=widgets.Layout(width='auto')
                        )
                        checkbox_index_map[len(checkboxes)] = i
                        checkboxes.append(cb)

                    delete_btn = widgets.Button(description='确认删除选中节点', button_style='danger', icon='trash')
                    no_delete_btn = widgets.Button(description='不删除，直接保存', button_style='info', icon='check')

                    def on_delete(btn):
                        if delete_result['confirmed']:
                            return
                        delete_result['deleted'] = {checkbox_index_map[i] for i, cb in enumerate(checkboxes) if
cb.value}
                        delete_result['confirmed'] = True
                        delete_btn.disabled = True
                        no_delete_btn.disabled = True
                        for cb in checkboxes:
                            cb.disabled = True
                        with delete_output:
                            clear_output()
                            if delete_result['deleted']:
                                print(f"  将删除 {len(delete_result['deleted'])} 个节点: {sorted(delete_result['deleted'])}")
                            else:
                                print("  未选择任何节点")

                    def on_no_delete(btn):
                        if delete_result['confirmed']:
                            return
                        delete_result['confirmed'] = True
                        delete_btn.disabled = True
                        no_delete_btn.disabled = True
                        for cb in checkboxes:
                            cb.disabled = True
                        with delete_output:
                            clear_output()
                            print("  保留全部节点")

                    delete_btn.on_click(on_delete)
                    no_delete_btn.on_click(on_no_delete)

                    print("\n  配体碎片可视化 (元素着色), 勾选要删除的配体碎片:")
                    frag_view.show()
                    mid = (len(checkboxes) + 1) // 2
                    display(widgets.HBox([widgets.VBox(checkboxes[:mid]), widgets.VBox(checkboxes[mid:])]))
                    display(widgets.HBox([delete_btn, no_delete_btn]))
                    display(delete_output)

                    print("  请选择要删除的节点...")
                    with ui_events() as poll:
                        while not delete_result['confirmed']:
                            poll(10)
                            time.sleep(0.1)

                    # --- 阶段 S7: 应用删除 + 保存 ---
                    if delete_result['deleted']:
                        combined_data = utils.remove_nodes_from_pyg_data(combined_data, delete_result['deleted'])
                        if combined_data is None:
                            raise ValueError("删除后无剩余节点")
                        print(f"  [Survival] 删除后: {combined_data.x.size(0)} 节点, {combined_data.edge_index.size(1)} 边")

                    # 保存 PyG Data
                    out_dir = os.path.join(TASK_A_OUTPUT_DIR, complex_id)
                    os.makedirs(out_dir, exist_ok=True)
                    torch.save(combined_data, os.path.join(out_dir, 'complex.pt'))

                    # 保存归中后的口袋 PDB
                    centered_pdb_lines = []
                    for line in pocket_pdb_lines:
                        try:
                            x = float(line[30:38]) - meb_center[0]
                            y = float(line[38:46]) - meb_center[1]
                            z = float(line[46:54]) - meb_center[2]
                            new_line = f"{line[:30]}{x:8.3f}{y:8.3f}{z:8.3f}{line[54:]}"
                            centered_pdb_lines.append(new_line)
                        except:
                            centered_pdb_lines.append(line)

                    with open(os.path.join(out_dir, 'pocket.pdb'), 'w') as f:
                        f.writelines(centered_pdb_lines)
                        f.write('END\n')
                    
                    _ligand_mol = utils.build_ligand_mol_from_pdb_lines(selected_ligand['atom_lines'])
                    if _ligand_mol is not None:
                        norm_ligand = Chem.Mol(_ligand_mol) # 建立副本
                        conf = norm_ligand.GetConformer()
                        for i in range(norm_ligand.GetNumAtoms()):
                            pos = np.array(conf.GetAtomPosition(i))
                            new_pos = pos - meb_center
                            conf.SetAtomPosition(i, (float(new_pos[0]), float(new_pos[1]), float(new_pos[2])))
                        
                        sdf_out_path = os.path.join(out_dir, f"{complex_id}_ligand_normalized.sdf")
                        with Chem.SDWriter(sdf_out_path) as writer:
                            writer.write(norm_ligand)
                        print(f"  [Survival] ✅ 已保存归一化配体: {sdf_out_path}")


                    centered_full_pdb_lines = []
                    for line in protein_lines:
                        try:
                            # 按照 PDB 固定列宽截取坐标并平移
                            x = float(line[30:38]) - meb_center[0]
                            y = float(line[38:46]) - meb_center[1]
                            z = float(line[46:54]) - meb_center[2]
                            new_line = f"{line[:30]}{x:8.3f}{y:8.3f}{z:8.3f}{line[54:]}"
                            centered_full_pdb_lines.append(new_line)
                        except:
                            # 如果该行格式异常（非标准 ATOM 行），原样保留
                            centered_full_pdb_lines.append(line)

                    full_protein_path = os.path.join(out_dir, 'full_protein_normalized.pdb')
                    with open(full_protein_path, 'w') as f:
                        f.writelines(centered_full_pdb_lines)
                        f.write('END\n')
                        
                    print(f"  ✅ 已保存归一化完整受体: {full_protein_path}")
                        
                    print(f"  [Survival] ✅ 已保存至: {out_dir}/")
                    success_count_a += 1

        except Exception as e:
            import traceback
            print(f"  [Error] {complex_id}: {e}")
            traceback.print_exc()
            fail_count_a += 1
            continue

    print(f"\n[{TASK_MODE}] 处理完成: 成功 {success_count_a}, 失败 {fail_count_a}")


In [ ]:
# =============================================================================
# 模块 7: 最终汇总与资源保存
# =============================================================================

print("\n" + "="*50)
print("开始执行最终汇总...")

cnt_success_a = locals().get('success_count_a', 0)
cnt_fail_a    = locals().get('fail_count_a', 0)

print("-" * 30)
print("PhiSSeparator 处理流程统计报告:")
print(f"  [{TASK_MODE}] 处理 : 成功 {cnt_success_a:4d} | 失败 {cnt_fail_a:4d}")
print("-" * 30)
print(f"  总计处理文件        : {cnt_success_a + cnt_fail_a}")
print(f"  总成功              : {cnt_success_a}")
print(f"  总失败              : {cnt_fail_a}")

print("\n正在保存词汇表状态...")
if 'vocab_data' in locals() and vocab_data:
    vocabulary.update(vocab_data.keys())
    save_vocabulary_state(vocabulary, vocab_data, VOCAB_FILE)
else:
    print("警告: 词汇表数据为空或未定义，跳过保存。")

print("="*50)
print("所有流程执行完毕。")

In [ ]:
import os
import numpy as np
from rdkit import Chem
# --- 执行区 ---
output_eval_dir = "./evaluation_data" 

# 新增这一行：手动指定一下当前任务的名字（随便起，用来做文件名的前缀）
task_id = "my_test_pocket"  # 如果你知道真实的口袋ID，比如 "1a2b"，可以替换掉这里
# ====================================================================
# [临时数据导出]：归一化并保存真实配体 (Ground Truth Ligand)
# ====================================================================
# 请确保当前 Notebook 内存中已有以下三个变量，如果没有请替换为你的实际变量名：
# 1. ligand_mol  -> 你选中的那个原始 RDKit 配体分子对象
# 2. meb_center  -> 你计算出的蛋白口袋/复合物中心坐标 (shape: [3,])
# 3. task_id     -> 你的任务编号，例如 "pocket_001"
# ====================================================================

def save_normalized_gt_ligand(mol, center_coords, save_dir, task_id):
    if mol is None:
        print("❌ 配体分子不存在！")
        return
        
    # 深拷贝一个分子，防止修改坐标污染后续的 Notebook 流程
    norm_ligand = Chem.Mol(mol)
    conf = norm_ligand.GetConformer()
    
    # 遍历所有原子，将绝对坐标减去 meb_center (归一化到原点附近)
    for i in range(norm_ligand.GetNumAtoms()):
        original_pos = np.array(conf.GetAtomPosition(i))
        new_pos = original_pos - center_coords
        conf.SetAtomPosition(i, (float(new_pos[0]), float(new_pos[1]), float(new_pos[2])))
        
    # 设定保存路径 (可以保存在 TASK_A 目录下，或者你的对比分析目录)
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"{task_id}_GT_ligand_normalized.sdf")
    
    # 保存为 SDF 文件
    try:
        with Chem.SDWriter(save_path) as writer:
            writer.write(norm_ligand)
        print(f"✅ 真实配体已归一化并保存至: {save_path}")
    except Exception as e:
        print(f"❌ 保存失败: {e}")

# --- 执行区 ---
# 假设你想保存在当前目录下的 evaluation 文件夹
output_eval_dir = "./evaluation_data" 

# 执行保存
save_normalized_gt_ligand(ligand_mol, meb_center, output_eval_dir, task_id)